# 3 · Volatility prediction

Trains and evaluates the two-stage model of the paper (Section IV-B) and its ablation variants:

1. **Stage 1: Past Volatility.** A 2-layer LSTM (64 units) and an MLP head, trained on the 29 historical-volatility features $v_{past}$; the checkpoint with the lowest validation loss is kept and **frozen**.
2. **Stage 2: QA residual.** QA embeddings → projection (3584 → 512 → 256) → BiLSTM (128 × 2) → masked additive attention → reduction (→ 64); the frozen volatility representation and the QA representation predict a correction $\Delta\hat y$, and $\hat y = \hat y^{base} + \Delta\hat y$.

AdamW (lr $10^{-4}$, weight decay $10^{-4}$), batch 32, 100 epochs, gradient clipping 1.0, best-validation checkpoint; in stage 2 the learning rate is halved on a validation plateau (patience 10, minimum $10^{-6}$). Every configuration is trained with the same **55 seeds**.

| Dataset | Train / dev | Test | Configurations |
|---|---|---|---|
| MAEC-15 | MAEC-15 | MAEC-15 (154 calls) | Past Volatility, Proposed (Table I) |
| MAEC-16 | MAEC-16 | MAEC-16 (280 calls) | all eight rows of Table III (and Table I) |
| EC | MAEC-15 + MAEC-16 | EC (110 calls, recomputed inputs) | Past Volatility, Proposed (Table I) |

**Paper's runs:** Google Colab, NVIDIA L4, PyTorch 2.11. The notebook is designed for Google Colab; it can also be run on your own machine (README, "Where the notebooks run"). On other hardware the numbers can differ slightly, as they can between GPUs on Colab. The released predictions (`results/predictions/`) are the exact record of the paper's numbers.

**Inputs:** the 19 embedding files in `WORK_DIR/embeddings/` (the data package: set `DOWNLOAD_PUBLIC_DATA = True` in the settings cell to have them downloaded and verified, or put them there yourself; or notebook 2's output); the KeFVP volatility files (downloaded from GitHub); `data/ec/ec_test_corrected.csv` from this repository. **Outputs:** `WORK_DIR/results/item_losses.csv` (every test prediction) and our results in Tables I and III in `WORK_DIR/results/tables/`.

### How to run

1. Edit the **settings cell** below if needed (run mode, folders). It is the only cell to edit.
2. Run all cells.

| `RUN_MODE` | What it does | Time |
|---|---|---|
| `"smoke"` | Tests your setup: runs every cell with 2 seeds and 2 epochs. **The numbers are meaningless.** Ends with `SMOKE TEST PASSED`. | minutes |
| `"check"` | Trains 3 of the paper's seeds in full (Past Volatility on all datasets, the proposed model on `CHECK_DATASETS`: MAEC-15 by default) and compares every prediction with the released one. | about 15 minutes on a GPU with the defaults |
| `"full"` | The paper's run: 12 models (Past Volatility on the 3 datasets and 9 QA-based models) × 55 seeds. | several GPU hours |

Run `"smoke"` first on a new setup. Each mode writes to its own results folder. A run that is interrupted starts again from the beginning.

Cell labels: **[SETUP]** folders, downloads, run mode (no model or training code) · **[CHECK]** verifies files or results; changes no input and no result (the `check` mode writes its report) · **[EXPERIMENT]** the paper's code, unchanged; do not edit it when reproducing the paper.

## 0. Setup

In [ ]:
# [SETUP] Settings: the only cell to edit. Nothing here changes the models or the training.

RUN_MODE = "full"
#   "full"  : the paper's run, 12 models x 55 seeds (several GPU hours)
#   "smoke" : tests your setup: every cell, 2 seeds x 2 epochs; the numbers are meaningless
#   "check" : 3 seeds in full, compared with the released predictions for the same seeds

WORK_DIR = None
#   The folder for the inputs and the results. None = the default:
#     on Colab : /content/drive/MyDrive/qa-grounded-volatility   (Google Drive, as in the paper's runs)
#     elsewhere: qa-grounded-volatility-work, in the folder this notebook runs in
#   Any folder works. On Colab, a folder outside /content/drive is deleted when the session ends.

EMBEDDING_DIR = None   # None = WORK_DIR/embeddings (the 19 files of the data package, or notebook 2's output)
OUTPUT_DIR = None      # None = WORK_DIR/results    ("smoke" and "check" write to a subfolder of it)

DOWNLOAD_PUBLIC_DATA = False
#   True : the setup downloads the 19 released embedding files (about 0.9 GB) from the public data package into
#          EMBEDDING_DIR and verifies each against its SHA-256. It never replaces a file that is already there.
#          On Colab the default folder is on your Google Drive, so the files take about 0.9 GB of its space.
#   False: you put the files there yourself (README, section "Data"), or they come from notebook 2.

CHECK_SEEDS = [1, 3, 5]        # "check" mode: any of the paper's 55 seeds
CHECK_DATASETS = ["maec15"]    # "check" mode: datasets for the proposed model, from "maec15", "maec16", "ec"

# Each setting can also be given as an environment variable (for batch jobs); a variable that is set wins over this cell:
# QAV_RUN_MODE, QAV_WORK_DIR, QAV_EMBEDDING_DIR, QAV_OUTPUT_DIR, QAV_DOWNLOAD_PUBLIC_DATA (1 or 0), and for "check" mode
# QAV_CHECK_SEEDS and QAV_CHECK_DATASETS (comma-separated, for example 1,3,5 and maec15,maec16,ec).

In [ ]:
# [SETUP] Environment: resolves the folders, mounts Google Drive if a folder is on it, finds the repository.
import os
import subprocess
import sys

RUN_MODE = os.environ.get("QAV_RUN_MODE") or RUN_MODE
if os.environ.get("QAV_DOWNLOAD_PUBLIC_DATA"):
    DOWNLOAD_PUBLIC_DATA = os.environ["QAV_DOWNLOAD_PUBLIC_DATA"].strip().lower() in ("1", "true", "yes")
if os.environ.get("QAV_CHECK_SEEDS"):
    CHECK_SEEDS = [int(s) for s in os.environ["QAV_CHECK_SEEDS"].split(",")]
if os.environ.get("QAV_CHECK_DATASETS"):
    CHECK_DATASETS = [s.strip() for s in os.environ["QAV_CHECK_DATASETS"].split(",")]
assert RUN_MODE in ("full", "smoke", "check"), f"RUN_MODE must be full, smoke or check, not {RUN_MODE!r}"
assert CHECK_DATASETS and set(CHECK_DATASETS) <= {"maec15", "maec16", "ec"}, f"CHECK_DATASETS: use maec15, maec16, ec, not {CHECK_DATASETS!r}"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

WORK_DIR = os.environ.get("QAV_WORK_DIR", WORK_DIR) or (
    "/content/drive/MyDrive/qa-grounded-volatility" if IN_COLAB else "qa-grounded-volatility-work")
EMBEDDING_DIR = os.environ.get("QAV_EMBEDDING_DIR", EMBEDDING_DIR) or os.path.join(WORK_DIR, "embeddings")
_results_root = os.environ.get("QAV_OUTPUT_DIR", OUTPUT_DIR) or os.path.join(WORK_DIR, "results")
# "smoke" and "check" never write into the folder of a full run
RESULTS_DIR_DRIVE = _results_root if RUN_MODE == "full" else os.path.join(_results_root, RUN_MODE)

_folders = [os.path.abspath(p) for p in (WORK_DIR, EMBEDDING_DIR, RESULTS_DIR_DRIVE)]
if IN_COLAB and any(p.startswith("/content/drive") for p in _folders):
    from google.colab import drive
    drive.mount("/content/drive")
WORK_DIR, EMBEDDING_DIR, RESULTS_DIR_DRIVE = _folders

# The repository: this notebook's own copy if it runs inside one, otherwise a fresh clone.
REPO_URL = "https://github.com/hosein97/ecc-volatility-prediction"
REPO_DIR = next((os.path.abspath(p) for p in (".", "..", "repo")
                 if os.path.exists(os.path.join(p, "scripts", "make_tables.py"))), None)
if REPO_DIR is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "repo"], check=True)
    REPO_DIR = os.path.abspath("repo")

print("run mode   :", RUN_MODE + (" (from the environment variable QAV_RUN_MODE)" if os.environ.get("QAV_RUN_MODE") else ""))
print("embeddings :", EMBEDDING_DIR)
print("results    :", RESULTS_DIR_DRIVE)
print("repository :", REPO_DIR)
print("environment:", "Google Colab" if IN_COLAB else "not Colab")
if RUN_MODE == "smoke":
    print("\n*** SMOKE TEST: 2 seeds, 2 epochs. This tests the setup. The numbers are meaningless. ***")
if IN_COLAB and not RESULTS_DIR_DRIVE.startswith("/content/drive") and RUN_MODE == "full":
    print("\nWARNING: the results folder is on the Colab runtime and is deleted when the session ends.")
if os.path.exists(os.path.join(RESULTS_DIR_DRIVE, "item_losses.csv")):
    print("\nNOTE: the results folder holds item_losses.csv from an earlier run; this run replaces it at the end.")

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
import os
import pickle
import random
from collections import defaultdict
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

DATA_DIR = Path(EMBEDDING_DIR)
RESULTS_DIR = Path(RESULTS_DIR_DRIVE)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [1, 3, 5, 6, 9, 11, 13, 14, 15, 17, 18, 19, 21, 23, 24, 26, 28, 30, 31, 32, 33, 34, 35, 39, 40, 42, 43, 44, 45, 46, 48, 50, 53, 59, 60, 61, 63, 64, 65, 66, 67, 69, 71, 73, 76,80, 81, 83, 87, 88, 89, 93, 94, 96, 98]

BATCH_SIZE = 32
QA_EMBED_DIM = 3584
HORIZONS = [3, 7, 15, 30]
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
SCHED_PATIENCE = 10
SCHED_FACTOR = 0.5
SCHED_MIN_LR = 1e-6
EARLY_STOP_PATIENCE = 100
NUM_EPOCHS = 100


def get_past_cols(dataset_key: str, split : str = "train"):
    """The 29 historical-volatility features v_1..v_29 (MAEC definition; the EC test file holds the
    corrected EC features in the same columns)."""
    return [f"past_{i}" for i in range(1, 30)]  # past_1 .. past_29


target_cols = [f"future_{h}" for h in HORIZONS]

In [ ]:
# [SETUP] Run mode. In "full" mode this cell changes nothing.
if RUN_MODE == "smoke":
    SEEDS, NUM_EPOCHS = SEEDS[:2], 2
elif RUN_MODE == "check":
    assert set(CHECK_SEEDS) <= set(SEEDS), "a check seed is not one of the paper's 55 seeds"
    SEEDS = [s for s in SEEDS if s in CHECK_SEEDS]

print(f"run mode: {RUN_MODE} | seeds: {len(SEEDS)} | epochs: {NUM_EPOCHS} | device: {device}")

## 1. Volatility labels and historical features

MAEC: the files released with KeFVP (pinned commit). EC: the KeFVP test calls with the historical features recomputed with the MAEC definition (`scripts/build_ec_features.py`; paper, Section IV-B); the targets are the released ones.

In [ ]:
# [SETUP] Download the KeFVP volatility files (pinned commit) into the folder this notebook runs in.
import urllib.request

_KEFVP = "https://raw.githubusercontent.com/hankniu01/KeFVP/95993892d087599dc5530804aada73f7b5004f39/price_data/maec"
for _year in ("15", "16"):
    for _split in ("train", "dev", "test"):
        _name = f"maec{_year}_{_split}_avg_val.csv"
        if not os.path.exists(_name):
            urllib.request.urlretrieve(f"{_KEFVP}/{_year}/{_name}", _name + ".part")
            os.replace(_name + ".part", _name)
print("KeFVP volatility files: ready")

In [ ]:
# [EXPERIMENT] The paper's code; only the path of the EC file changed (it follows REPO_DIR).
maec15_train_avg_val_df = pd.read_csv("maec15_train_avg_val.csv")
maec15_dev_avg_val_df = pd.read_csv("maec15_dev_avg_val.csv")
maec15_test_avg_val_df = pd.read_csv("maec15_test_avg_val.csv")

maec16_train_avg_val_df = pd.read_csv("maec16_train_avg_val.csv")
maec16_dev_avg_val_df = pd.read_csv("maec16_dev_avg_val.csv")
maec16_test_avg_val_df = pd.read_csv("maec16_test_avg_val.csv")

ec_test_avg_val_df = pd.read_csv(os.path.join(REPO_DIR, "data", "ec", "ec_test_corrected.csv"))   # 110 calls, corrected inputs

maec_combined_train_df = pd.concat(
    [maec15_train_avg_val_df, maec16_train_avg_val_df], ignore_index=True
)
maec_combined_dev_df = pd.concat(
    [maec15_dev_avg_val_df, maec16_dev_avg_val_df], ignore_index=True
)

PRICE_DFS = {
    "maec15": {
        "train": maec15_train_avg_val_df,
        "dev": maec15_dev_avg_val_df,
        "test": maec15_test_avg_val_df,
    },
    "maec16": {
        "train": maec16_train_avg_val_df,
        "dev": maec16_dev_avg_val_df,
        "test": maec16_test_avg_val_df,
    },
    "ec": {
        "train": maec_combined_train_df,
        "dev": maec_combined_dev_df,
        "test": ec_test_avg_val_df,
    },
}

## 2. QA embeddings

In [ ]:
# [SETUP] Optional: download the 19 released embedding files from the public data package and verify them.
# This cell downloads nothing unless DOWNLOAD_PUBLIC_DATA = True in the settings cell.
import hashlib
import shutil
import time

DATA_PACKAGE_URL = "https://drive.google.com/drive/folders/1LOMRqqFQrA-Oz4n6GBDxlLWOfPE8VxZp"
# file -> SHA-256 of the released file (the manifest; docs/DATA.md lists the same values)
RELEASED_EMBEDDINGS = {
    "kefvp_dev_maec15_qa_text_audio_prompt2_embeddings.pkl": "df70402a856e90fdf3919b79a128256a4f908428e63c136f029ecff6630770d4",
    "kefvp_dev_maec16_audio_embeddings.pkl": "3cb99daf22f375a2384007733e4f90b9b8d43073f234a42794a073b7ef33ca77",
    "kefvp_dev_maec16_chunk_text_audio_embeddings.pkl": "1850ff3dbb15202af5902b09bcffc0ef652e6267896905b0fcfa56fb5d5c45c6",
    "kefvp_dev_maec16_embeddings.pkl": "82bc523bc456fabbe9f1c68cf98b6dedca2a521f03479d858397e874ee131f3e",
    "kefvp_dev_maec16_qa_text_audio_prompt2_embeddings.pkl": "21c702b926af72e6fcbb3993898d9c4c0e013153ec439c210ecb824338665b95",
    "kefvp_dev_maec16_text_embeddings.pkl": "495d9b7a7f14b87be45911488348c8720553f401e72c7715a5661bdbbf2a3c4a",
    "kefvp_test_ec_qa_text_audio_prompt2_embeddings.pkl": "226e28c2ea362c66a60f6d0d6337b87a7ffbf124a00941a613376ce1cb54de10",
    "kefvp_test_maec15_qa_text_audio_prompt2_embeddings.pkl": "cb37cb0a12ad136882c3478cc37c2c65a4e8b5d9791425833109684f00f9f526",
    "kefvp_test_maec16_audio_embeddings.pkl": "3f5d0d51116d31cbc410ec7c85dde609c07e9239dc4482478f06393b8777ef1d",
    "kefvp_test_maec16_chunk_text_audio_embeddings.pkl": "cb7a4d8cf90d0a49545d632aa82cc97a0f2d339ee06fcaa907daa4d46e983308",
    "kefvp_test_maec16_embeddings.pkl": "da9e369e7c9b0506c01a38d558bf683392a5f58bdab6c8868ebc96f8b3a71eff",
    "kefvp_test_maec16_qa_text_audio_prompt2_embeddings.pkl": "23af33389762c58fa10ea3808a6fb8e65c59f6e2c78c93d1bfd0638d35e49d3e",
    "kefvp_test_maec16_text_embeddings.pkl": "e2ce2570b5fdf070ef5fbddce34b109154951ed20faba5dcdec00b4f1110624f",
    "kefvp_train_maec15_qa_text_audio_prompt2_embeddings.pkl": "04056abd9bef62eb22b3031631bd08acaa08c26ba34d5381c1b364dd4c6af9fb",
    "kefvp_train_maec16_audio_embeddings.pkl": "b3918575c77a2b1f0a15592763479a594a65019cd5c1dc867a7cea1203e1a55f",
    "kefvp_train_maec16_chunk_text_audio_embeddings.pkl": "c10201c93b59e3604eb0eeda77cb7c51d15657639ba6149d643e772b3de69ff6",
    "kefvp_train_maec16_embeddings.pkl": "b2f64b2cfd00d1fd74b2bdbd7ef52436ab250c31b979b59b97705ce5f87b846b",
    "kefvp_train_maec16_qa_text_audio_prompt2_embeddings.pkl": "892aaabf9a182647ba0d22f16244303f8079a50bb7cf3b061d8ab0f34cabadb0",
    "kefvp_train_maec16_text_embeddings.pkl": "5940eab89f718527619b9397349b4fc651b7664009c8b8a7eedc0b8f2f6e7073"
}
# file -> id of the file in the public Google Drive folder
_DRIVE_IDS = {
    "kefvp_dev_maec15_qa_text_audio_prompt2_embeddings.pkl": "1u6XyLc-4t1bPVZ3SCHYNUEppkKJBhv_0",
    "kefvp_dev_maec16_audio_embeddings.pkl": "1DdcMnbtVOreNlO9wk9oamMQ70ZaufcVg",
    "kefvp_dev_maec16_chunk_text_audio_embeddings.pkl": "1JHYM1gxm8pBBIlgcF1CZ89Jg_M1qJrML",
    "kefvp_dev_maec16_embeddings.pkl": "1p446HrSwBTaxzRmtm6nq4Em9CBTtPzoy",
    "kefvp_dev_maec16_qa_text_audio_prompt2_embeddings.pkl": "1TOxEDhwTX2YZpIP6lVQQh_2vLLTXD26n",
    "kefvp_dev_maec16_text_embeddings.pkl": "12Ubk9DMIJ6TRLE2bkdutpxFc6yqtiofo",
    "kefvp_test_ec_qa_text_audio_prompt2_embeddings.pkl": "1MgmWg4JnxVIXlv9DpBX5tGKftcEP4ttx",
    "kefvp_test_maec15_qa_text_audio_prompt2_embeddings.pkl": "1TeYhZ2vfwdZ8tclp5GAih8MjtYD70N17",
    "kefvp_test_maec16_audio_embeddings.pkl": "197y2iaQeAaGGiwKKV02FwPCE0Rqzu00j",
    "kefvp_test_maec16_chunk_text_audio_embeddings.pkl": "1Nd6APSaw8turhMSHVJAH3y6Ykf-W9ihM",
    "kefvp_test_maec16_embeddings.pkl": "1SMy2dtacRWeM857_crpyHpeNJ3VkhOxu",
    "kefvp_test_maec16_qa_text_audio_prompt2_embeddings.pkl": "1ARDi8QFEOudIRN1h0wt7pICBi7REwrCt",
    "kefvp_test_maec16_text_embeddings.pkl": "1JWPQonrkB21DtCo0z9twsKdzJgp0DVsw",
    "kefvp_train_maec15_qa_text_audio_prompt2_embeddings.pkl": "1XEP-QRLebFzkL2o3A16EM60GHkgMzq4Z",
    "kefvp_train_maec16_audio_embeddings.pkl": "1i5fq1wJuDEgjiX1ncAHfEzKS0UqsrKgG",
    "kefvp_train_maec16_chunk_text_audio_embeddings.pkl": "1FoFd-RSMqjm8TABXFTj9d3U6-52fNiAi",
    "kefvp_train_maec16_embeddings.pkl": "1OS4vRRi20GgjnOldElr52vXdRTOhrzMx",
    "kefvp_train_maec16_qa_text_audio_prompt2_embeddings.pkl": "1ntt_rb-6KLzFhpQdqes8DKfB7AcocXR2",
    "kefvp_train_maec16_text_embeddings.pkl": "1mSwwuHhDLXQ39-TNoQbB8xE8_wv0O8NJ"
}
_MANUAL = (f"Download the embeddings/ folder by hand from {DATA_PACKAGE_URL} and put its 19 files in {EMBEDDING_DIR} "
           "(README, section 'Data'). Google Drive limits how often a file can be downloaded; an automatic "
           "download that is refused now usually works again some hours later.")
_VERIFIED = set()   # files whose SHA-256 was checked in this session


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 22), b""):
            h.update(block)
    return h.hexdigest()


def _is_released(path, name):
    ok = _sha256(path) == RELEASED_EMBEDDINGS[name]
    if ok:
        _VERIFIED.add(name)
    return ok


print("data package:", DATA_PACKAGE_URL, "| embeddings/: 19 files, about 0.9 GB")
if not DOWNLOAD_PUBLIC_DATA:
    print("automatic download: off (DOWNLOAD_PUBLIC_DATA = False in the settings cell)")
else:
    _missing = [n for n in RELEASED_EMBEDDINGS if not os.path.exists(os.path.join(EMBEDDING_DIR, n))]
    _different = [n for n in RELEASED_EMBEDDINGS
                  if n not in _missing and not _is_released(os.path.join(EMBEDDING_DIR, n), n)]
    if _different:
        raise RuntimeError(
            f"{len(_different)} files in {EMBEDDING_DIR} are not the released files (changed, incomplete, or "
            "embeddings you computed with notebook 2):\n  " + "\n  ".join(_different)
            + "\nNothing was downloaded and nothing was changed. Move these files away, or set EMBEDDING_DIR to "
              "another folder, and run this cell again. To use your own embeddings instead, set "
              "DOWNLOAD_PUBLIC_DATA = False.")
    if not _missing:
        print("automatic download: all 19 released files are already in place and verified; nothing to download")
    else:
        try:
            import gdown
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
            import gdown
        # the files reach EMBEDDING_DIR only after every one of them is verified
        _tmp = os.path.join(os.path.dirname(EMBEDDING_DIR), os.path.basename(EMBEDDING_DIR) + ".download")
        os.makedirs(_tmp, exist_ok=True)
        try:
            print(f"automatic download: {len(_missing)} of the 19 files are missing; downloading them to {_tmp}")
            for _name in _missing:
                _part = os.path.join(_tmp, _name)
                if os.path.exists(_part):
                    if _sha256(_part) == RELEASED_EMBEDDINGS[_name]:
                        continue                      # complete, from a run that was interrupted later
                    os.remove(_part)
                _why = "no file was written"
                for _attempt in range(3):             # a refused download often works a moment later
                    try:
                        gdown.download(id=_DRIVE_IDS[_name], output=_part, quiet=False)
                    except Exception as e:
                        _why = f"{type(e).__name__}: {str(e).strip()[:300]}"
                    if os.path.exists(_part):
                        break
                    time.sleep(15)
                else:
                    raise RuntimeError(f"the download of {_name} failed three times ({_why}). "
                                       f"Nothing was put into {EMBEDDING_DIR}. " + _MANUAL)
            _bad = [n for n in _missing if _sha256(os.path.join(_tmp, n)) != RELEASED_EMBEDDINGS[n]]
            if _bad:
                for _name in _bad:
                    os.remove(os.path.join(_tmp, _name))
                raise RuntimeError(
                    f"{len(_bad)} downloaded files do not match the released SHA-256 values:\n  " + "\n  ".join(_bad)
                    + f"\nThey were deleted, and nothing was put into {EMBEDDING_DIR}. Run this cell again. " + _MANUAL)
            os.makedirs(EMBEDDING_DIR, exist_ok=True)
            for _name in _missing:
                _target = os.path.join(EMBEDDING_DIR, _name)
                if os.path.exists(_target):
                    raise RuntimeError(f"{_target} appeared during the download; it was not replaced. Run this cell again.")
                shutil.move(os.path.join(_tmp, _name), _target)
                _VERIFIED.add(_name)
        finally:
            if not os.listdir(_tmp):          # also after a failure: no empty folder is left behind
                os.rmdir(_tmp)
        print(f"automatic download: {len(_missing)} files downloaded and verified; "
              f"all 19 released files are in {EMBEDDING_DIR}")

In [ ]:
# [CHECK] The 19 embedding files are in EMBEDDING_DIR. Changes nothing.
_missing = [n for n in RELEASED_EMBEDDINGS if not os.path.exists(os.path.join(EMBEDDING_DIR, n))]
if _missing:
    raise FileNotFoundError(
        f"{len(_missing)} of the 19 embedding files are not in {EMBEDDING_DIR}:\n  " + "\n  ".join(_missing)
        + "\nSet DOWNLOAD_PUBLIC_DATA = True in the settings cell to download them, or put the embeddings/ folder "
          "of the data package there yourself (README, section 'Data'), or run notebook 2, or point EMBEDDING_DIR "
          "in the settings cell to the folder that holds them.")

_other = [n for n in RELEASED_EMBEDDINGS
          if n not in _VERIFIED and not _is_released(os.path.join(EMBEDDING_DIR, n), n)]
print(f"embedding files: 19 found, {19 - len(_other)} identical to the released files (SHA-256)")
if _other:
    print("NOTE: these files are not byte-identical to the released ones. This is expected if you computed them with")
    print("      notebook 2, or ran notebook 2's section F on this folder (it saves every file again, with the same")
    print("      content). Otherwise download them again:")
    for _name in _other:
        print("   ", _name)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
MODALITY_SUFFIX = {
    "text_audio_prompt1": "embeddings",
    "text_only": "text_embeddings",
    "text_audio_prompt2": "qa_text_audio_prompt2_embeddings",
    "audio_only": "audio_embeddings",
    "chunk_text_audio": "chunk_text_audio_embeddings",
}

MODALITIES = list(MODALITY_SUFFIX)


def load_embeddings(dataset: str, split: str, modality: str):
    """Load a single embeddings pickle file.

    dataset: "maec15" | "maec16" | "ec"
    split:   "train" | "dev" | "test"  (EC only has "test")
    modality: one of MODALITY_SUFFIX
    """
    suffix = MODALITY_SUFFIX[modality]
    fname = f"kefvp_{split}_{dataset}_{suffix}.pkl"
    path = DATA_DIR / fname
    with open(path, "rb") as f:
        return pickle.load(f)


# The 19 files behind the paper's configurations: the Volatility Prompt embeddings for every
# dataset, and the four ablation variants for MAEC-16 only (Table III).
# embeddings[(dataset, split, modality)] -> list[dict] with "ticker"/"date"/"embedding"
EMBEDDINGS = {}

for _dataset in ("maec15", "maec16"):
    _modalities = MODALITIES if _dataset == "maec16" else ["text_audio_prompt2"]
    for _split in ("train", "dev", "test"):
        for _modality in _modalities:
            EMBEDDINGS[(_dataset, _split, _modality)] = load_embeddings(
                _dataset, _split, _modality
            )

EMBEDDINGS[("ec", "test", "text_audio_prompt2")] = load_embeddings("ec", "test", "text_audio_prompt2")

print(f"Loaded {len(EMBEDDINGS)} embedding files.")

In [ ]:
# [CHECK] Every embedding record carries its ticker and date. Changes nothing.
_untagged = sorted("/".join(k) for k, v in EMBEDDINGS.items() if v and "ticker" not in v[0])
if _untagged:
    raise RuntimeError(f"embedding files without ticker/date: {_untagged}. Run section F of notebook 2.")

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
def build_lookup(embedding_lists):
    """Index a list of embedding-lists by (ticker, date) -> list[np.ndarray]."""
    lookup = defaultdict(list)
    for dataset in embedding_lists:
        for item in dataset:
            key = (item["ticker"], str(item["date"]))
            lookup[key].append(item["embedding"].astype(np.float32))
    return lookup


def get_qa_lists(df, lookup, is_ec: bool = False):
    """For each row of a price/label dataframe, return the list of QA embeddings
     whose (ticker, date) matches that row."""
    result = []

    for _, row in df.iterrows():

        ticker = row["ticker"]

        if is_ec:
            date = f"{int(row['year']):04d}{int(row['month']):02d}{int(row['day']):02d}"
        else:
            date = str(row["time"]).replace("-", "")

        key = (ticker, date)
        qas = lookup.get(key, [])
        if len(qas) == 0:
            continue
        result.append(qas)

    return result




def get_data_and_qa_for(dataset_key: str, split: str, modality: str):
    """
    Return a price/label dataframe and its corresponding QA embeddings,
    aligned row-by-row.
    """

    df = PRICE_DFS[dataset_key][split].copy()
    if dataset_key == "ec":
        if split == "test":
            embedding_lists = [
                EMBEDDINGS[("ec", "test", modality)]
            ]
            is_ec = True
        else:
            embedding_lists = [
                EMBEDDINGS[("maec15", split, modality)],
                EMBEDDINGS[("maec16", split, modality)],
            ]
            is_ec = False
    else:
        embedding_lists = [
            EMBEDDINGS[(dataset_key, split, modality)]
        ]
        is_ec = False

    lookup = build_lookup(embedding_lists)

    rows = []
    qa_lists = []

    for _, row in df.iterrows():
        ticker = row["ticker"]

        if is_ec:
            date = (
                f"{int(row['year']):04d}"
                f"{int(row['month']):02d}"
                f"{int(row['day']):02d}"
            )
        else:
            date = str(row["time"]).replace("-", "")

        qas = lookup.get((ticker, date), [])

        if not qas:
            continue

        rows.append(row)
        qa_lists.append(qas)

    aligned_df = pd.DataFrame(rows).reset_index(drop=True)

    assert len(aligned_df) == len(qa_lists)

    return aligned_df, qa_lists

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
def get_item_ids(df: pd.DataFrame, is_ec: bool) -> pd.DataFrame:
    """ticker + normalized date string for each row of a test dataframe, aligned
    to df's row order. Mirrors the (ticker, date) key construction used in
    get_data_and_qa_for so ids stay consistent with how QA embeddings were
    looked up for the same rows."""
    if is_ec:
        date = (
            df["year"].astype(int).astype(str).str.zfill(4)
            + df["month"].astype(int).astype(str).str.zfill(2)
            + df["day"].astype(int).astype(str).str.zfill(2)
        )
    else:
        date = df["time"].astype(str).str.replace("-", "", regex=False)
    return pd.DataFrame({"ticker": df["ticker"].values, "date": date.values})

## 3. Datasets and models

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
class VolDataset(Dataset):
    """Volatility-history-only dataset, used for the vol-only baseline."""

    def __init__(self, df, past_cols):
        self.X = df[past_cols].values.astype(np.float32)
        self.y = df[target_cols].values.astype(np.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class ECCDataset(Dataset):
    """Volatility history + variable-length list of QA embeddings per example.

    Used for every fusion/QA-only experiment. The "whole chunk" (no-QA) modality
    reuses this same class -- each example's "QA list" there is just a length-1
    list containing the single chunk embedding.
    """

    def __init__(self, df, qa_lists, past_cols):
        self.X = df[past_cols].values.astype(np.float32)
        self.y = df[target_cols].values.astype(np.float32)
        self.qa_lists = qa_lists

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.qa_lists[idx], self.y[idx]


def collate_fn(batch):
    """Pad the variable-length QA lists in a batch to a common length and build a
    boolean mask marking real (non-padding) positions."""
    vols, qas, ys = zip(*batch)

    vols = torch.tensor(np.stack(vols), dtype=torch.float32)
    ys = torch.tensor(np.stack(ys), dtype=torch.float32)

    max_len = max(max(len(x), 1) for x in qas)

    qa_tensor = torch.zeros(len(qas), max_len, QA_EMBED_DIM, dtype=torch.float32)
    mask = torch.zeros(len(qas), max_len, dtype=torch.bool)

    for i, qa_list in enumerate(qas):
        if len(qa_list) == 0:
            continue
        arr = np.stack(qa_list)
        qa_tensor[i, : len(arr)] = torch.tensor(arr)
        mask[i, : len(arr)] = True

    return vols, qa_tensor, mask, ys

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
class AttentionPooling(nn.Module):
    """Masked attention pooling over a padded sequence of QA embeddings."""

    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x, mask):
        scores = self.attn(x).squeeze(-1)
        scores = scores.masked_fill(~mask, -1e9)

        weights = torch.softmax(scores, dim=1)
        weights = weights * mask.float()
        weights = weights / (weights.sum(dim=1, keepdim=True) + 1e-8)

        pooled = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return pooled

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
def run_packed_bilstm(lstm, x, mask):
    """Run `lstm` over a padded, batch-first sequence via pack_padded_sequence.

    Without packing, a bidirectional LSTM's backward pass starts at the padded
    tail and only reaches real tokens after stepping through the pad positions
    first, so its hidden state at every real position is contaminated by a
    batch-dependent number of leading pad steps. Packing removes those pad
    steps from the recurrence entirely; masking in AttentionPooling alone does
    not fix this since it only zeroes pooling weights after the LSTM has run.
    """
    lengths = mask.sum(dim=1).clamp(min=1).to("cpu")
    packed = nn.utils.rnn.pack_padded_sequence(
        x, lengths, batch_first=True, enforce_sorted=False
    )
    packed_out, _ = lstm(packed)
    out, _ = nn.utils.rnn.pad_packed_sequence(
        packed_out, batch_first=True, total_length=x.size(1)
    )
    return out

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
class VolOnlyModel(nn.Module):
    """Volatility-history-only baseline: 2-layer LSTM over `past_cols`,

    This is the frozen baseline whose `vol_lstm` + `head` get reused (frozen) by
    `VolECCResidualGatedModel`.
    """

    def __init__(self):
        super().__init__()

        self.vol_lstm = nn.LSTM(
            input_size=1, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2
        )

        self.head = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4),
        )

    def forward(self, vol):
        vol = vol.unsqueeze(-1)  # [B, 28] -> [B, 28, 1]
        _, (h, _) = self.vol_lstm(vol)
        vol_repr = h[-1]  # [B, 64]
        return self.head(vol_repr)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
class QAOnlyModel(nn.Module):
    """QA-embeddings-only model: no volatility branch at all."""

    def __init__(self):
        super().__init__()

        self.qa_proj = nn.Sequential(
            nn.Linear(QA_EMBED_DIM, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
        )

        self.qa_lstm = nn.LSTM(
            input_size=256, hidden_size=128, num_layers=1, batch_first=True, bidirectional=True
        )

        self.attention_pool = AttentionPooling(hidden_dim=256)

        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4),
        )

    def forward(self, qa, mask):
        qa = self.qa_proj(qa)
        qa_out = run_packed_bilstm(self.qa_lstm, qa, mask)
        qa_repr = self.attention_pool(qa_out, mask)
        return self.head(qa_repr)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
class VolECCConcatModel(nn.Module):
    """Concat fusion: vol_lstm (trained from scratch) || (qa_proj -> BiLSTM ->
    AttentionPooling), concatenated and passed through an MLP head."""

    def __init__(self):
        super().__init__()

        self.vol_lstm = nn.LSTM(
            input_size=1, hidden_size=64, num_layers=2, batch_first=True, dropout=0.2
        )

        self.qa_proj = nn.Sequential(
            nn.Linear(QA_EMBED_DIM, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
        )

        self.qa_lstm = nn.LSTM(
            input_size=256, hidden_size=128, num_layers=1, batch_first=True, bidirectional=True
        )

        self.attention_pool = AttentionPooling(hidden_dim=256)

        self.head = nn.Sequential(
            nn.Linear(64 + 256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4),
        )

    def forward(self, vol, qa, mask):
        vol = vol.unsqueeze(-1)
        _, (h, _) = self.vol_lstm(vol)
        vol_repr = h[-1]

        qa = self.qa_proj(qa)
        qa_out = run_packed_bilstm(self.qa_lstm, qa, mask)
        qa_repr = self.attention_pool(qa_out, mask)

        fused = torch.cat([vol_repr, qa_repr], dim=-1)
        out = self.head(fused)
        return out

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
class VolECCResidualGatedModel(nn.Module):
    """Residual/frozen fusion: a frozen, pretrained `VolOnlyModel` (`vol_lstm` +
    `head`, both `requires_grad=False`) predicts a base volatility forecast; a QA
    branch (proj -> BiLSTM -> AttentionPooling -> qa_reduce) predicts a delta on
    top of it. `delta_head`'s final layer is zero-initialized so the model starts
    out numerically identical to the frozen vol-only baseline and has to learn to
    deviate from it.
    """

    def __init__(self, vol_model: "VolOnlyModel"):
        super().__init__()

        # Reuse the pretrained volatility branch, frozen.
        self.vol_lstm = vol_model.vol_lstm
        self.vol_head = vol_model.head
        for p in self.vol_lstm.parameters():
            p.requires_grad = False
        for p in self.vol_head.parameters():
            p.requires_grad = False

        self.qa_proj = nn.Sequential(
            nn.Linear(QA_EMBED_DIM, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
        )

        self.qa_lstm = nn.LSTM(
            input_size=256, hidden_size=128, num_layers=1, batch_first=True, bidirectional=True
        )

        self.attention_pool = AttentionPooling(hidden_dim=256)

        self.qa_reduce = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
        )

        self.delta_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 4),
        )
        nn.init.zeros_(self.delta_head[-1].weight)
        nn.init.zeros_(self.delta_head[-1].bias)

    def train(self, mode: bool = True):
        # requires_grad=False above only blocks weight updates; it doesn't stop
        # dropout inside the frozen branch from being re-enabled by model.train().
        # Force it back to eval() every time so the frozen baseline stays
        # deterministic during training, not just during evaluation.
        super().train(mode)
        self.vol_lstm.eval()
        self.vol_head.eval()
        return self

    def forward(self, vol, qa, mask):
        vol = vol.unsqueeze(-1)
        _, (h, _) = self.vol_lstm(vol)
        vol_repr = h[-1]

        qa = self.qa_proj(qa)
        qa_out = run_packed_bilstm(self.qa_lstm, qa, mask)
        qa_repr = self.attention_pool(qa_out, mask)
        qa_repr = self.qa_reduce(qa_repr)

        vol_pred = self.vol_head(vol_repr)
        delta_input = torch.cat([vol_repr, qa_repr], dim=-1)
        delta = self.delta_head(delta_input)

        out = vol_pred + delta
        return out, vol_pred, delta


FUSION_MODEL_CLASSES = {
    "concat": VolECCConcatModel,
    "residual_gated": VolECCResidualGatedModel,
    "qa_only": QAOnlyModel,
}

## 4. Stage 1: Past Volatility (trained per seed, then frozen)

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
from tqdm.auto import tqdm
def train_vol_only_baseline(dataset_key: str, seeds=SEEDS):
    """Train VolOnlyModel once per seed for one dataset key, returning
    {seed: state_dict}, a per-seed results dataframe, and a long-format
    per-test-item loss dataframe (one row per seed x test item x horizon)."""

    train_df = PRICE_DFS[dataset_key]["train"]
    dev_df = PRICE_DFS[dataset_key]["dev"]
    test_df = PRICE_DFS[dataset_key]["test"]
    past_cols = get_past_cols(dataset_key)
    if dataset_key == 'ec':
      test_past_cols = get_past_cols(dataset_key, 'test')
    else:
      test_past_cols = past_cols

    item_ids = get_item_ids(test_df, is_ec=(dataset_key == "ec"))

    best_states = {}
    all_results = []
    all_item_results = []

    for seed in tqdm(seeds, desc=f"vol_only:{dataset_key}", unit="seed"):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        train_loader = DataLoader(VolDataset(train_df, past_cols), batch_size=BATCH_SIZE, shuffle=True)
        dev_loader = DataLoader(VolDataset(dev_df, past_cols), batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(VolDataset(test_df, test_past_cols), batch_size=BATCH_SIZE, shuffle=False)

        model = VolOnlyModel().to(device)
        criterion = nn.MSELoss()
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
        )

        best_val = float("inf")
        best_state = None
        counter = 0

        for epoch in range(NUM_EPOCHS):
            model.train()
            for vol, y in train_loader:
                vol, y = vol.to(device), y.to(device)
                optimizer.zero_grad()
                pred = model(vol)
                loss = criterion(pred, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for vol, y in dev_loader:
                    vol, y = vol.to(device), y.to(device)
                    val_loss += criterion(model(vol), y).item()
            val_loss /= len(dev_loader)

            if val_loss < best_val:
                best_val = val_loss
                best_state = deepcopy(model.state_dict())
                counter = 0
            else:
                counter += 1
                if counter >= EARLY_STOP_PATIENCE:
                    break

        best_states[seed] = best_state
        model.load_state_dict(best_state)

        model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for vol, y in test_loader:
                vol = vol.to(device)
                preds.append(model(vol).cpu().numpy())
                targets.append(y.numpy())
        preds = np.concatenate(preds)
        targets = np.concatenate(targets)

        mses = [
            mean_squared_error(targets[:, i], preds[:, i]) for i in range(len(HORIZONS))
        ]
        avg_mse = float(np.mean(mses))

        result = {"seed": seed, "avg_mse": avg_mse}
        for i, hrz in enumerate(HORIZONS):
            result[f"mse_{hrz}"] = mses[i]
        all_results.append(result)

        for i, hrz in enumerate(HORIZONS):
            all_item_results.append(pd.DataFrame({
                "seed": seed,
                "dataset": dataset_key,
                "modality": "vol_only",
                "fusion_type": "n/a",
                "ticker": item_ids["ticker"],
                "date": item_ids["date"],
                "horizon": hrz,
                "pred": preds[:, i],
                "target": targets[:, i],
                "sq_err": (preds[:, i] - targets[:, i]) ** 2,
            }))

        torch.save(best_state, RESULTS_DIR / f"vol_only_state_{dataset_key}_{seed}.pt")

    results_df = pd.DataFrame(all_results)
    results_df.to_csv(RESULTS_DIR / f"results_{dataset_key}_vol_only.csv", index=False)

    item_results_df = pd.concat(all_item_results, ignore_index=True)

    return best_states, results_df, item_results_df


VOL_ONLY_DATASET_KEYS = ["maec15", "maec16", "ec"]

best_models = {}
vol_only_results = {}
vol_only_item_results = {}

for _dataset_key in VOL_ONLY_DATASET_KEYS:
    _states, _results, _item_results = train_vol_only_baseline(_dataset_key)
    best_models[_dataset_key] = _states
    vol_only_results[_dataset_key] = _results
    vol_only_item_results[_dataset_key] = _item_results

## 5. Stage 2 and the other configurations

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
from tqdm.auto import tqdm
def run_experiment(
    name: str,
    dataset_key: str,
    modality: str,
    fusion_type: str,
    train_qas,
    dev_qas,
    test_qas,
    train_df,
    dev_df,
    test_df,
    needs_vol_pretrain: bool = True,
    seeds=SEEDS,
):
    """Train+evaluate one (dataset, modality, fusion_type) combination across all
    seeds, returning a per-seed results dataframe and a long-format per-test-item
    loss dataframe (one row per seed x test item x horizon); the results
    dataframe is also written to RESULTS_DIR.

    fusion_type: "concat"   | "residual_gated" | "qa_only"
    needs_vol_pretrain: True for "residual_gated" (loads the frozen VolOnlyModel
        weights trained in the previous section); ignored otherwise.
    """
    model_cls = FUSION_MODEL_CLASSES[fusion_type]
    past_cols = get_past_cols(dataset_key)
    if dataset_key == 'ec':
      test_past_cols = get_past_cols(dataset_key, 'test')
    else:
      test_past_cols = past_cols

    all_results = []
    all_item_results = []

    item_ids = get_item_ids(test_df, is_ec=(dataset_key == "ec"))

    for seed in tqdm(seeds, desc=name, unit="seed"):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

        train_loader = DataLoader(
            ECCDataset(train_df, train_qas, past_cols), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
        )
        dev_loader = DataLoader(
            ECCDataset(dev_df, dev_qas, past_cols), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
        )
        test_loader = DataLoader(
            ECCDataset(test_df, test_qas, test_past_cols), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
        )

        if fusion_type == "residual_gated" and needs_vol_pretrain:
            vol_model = VolOnlyModel().to(device)
            vol_model.load_state_dict(best_models[dataset_key][seed])
            model = model_cls(vol_model).to(device)
        else:
            model = model_cls().to(device)

        criterion = nn.MSELoss()
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=SCHED_FACTOR,
            patience=SCHED_PATIENCE,
            threshold=1e-4,
            min_lr=SCHED_MIN_LR,
        )

        best_val = float("inf")
        best_state = None
        counter = 0

        for epoch in range(NUM_EPOCHS):
            model.train()
            for vol, qa, mask, y in train_loader:
                vol, qa, mask, y = vol.to(device), qa.to(device), mask.to(device), y.to(device)
                optimizer.zero_grad()

                if fusion_type == "residual_gated":
                    pred, _, _ = model(vol, qa, mask)
                elif fusion_type == "qa_only":
                    pred = model(qa, mask)
                else:
                    pred = model(vol, qa, mask)

                loss = criterion(pred, y)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                optimizer.step()

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for vol, qa, mask, y in dev_loader:
                    vol, qa, mask, y = vol.to(device), qa.to(device), mask.to(device), y.to(device)
                    if fusion_type == "residual_gated":
                        pred, _, _ = model(vol, qa, mask)
                    elif fusion_type == "qa_only":
                        pred = model(qa, mask)
                    else:
                        pred = model(vol, qa, mask)
                    val_loss += criterion(pred, y).item()
            val_loss /= len(dev_loader)
            scheduler.step(val_loss)

            if val_loss < best_val:
                best_val = val_loss
                best_state = deepcopy(model.state_dict())
                counter = 0
            else:
                counter += 1
                if counter >= EARLY_STOP_PATIENCE:
                    break

        model.load_state_dict(best_state)
        model.eval()

        preds, targets = [], []
        with torch.no_grad():
            for vol, qa, mask, y in test_loader:
                vol, qa, mask = vol.to(device), qa.to(device), mask.to(device)
                if fusion_type == "residual_gated":
                    pred, _, _ = model(vol, qa, mask)
                elif fusion_type == "qa_only":
                    pred = model(qa, mask)
                else:
                    pred = model(vol, qa, mask)
                preds.append(pred.cpu().numpy())
                targets.append(y.numpy())

        preds = np.concatenate(preds)
        targets = np.concatenate(targets)

        mses = [
            mean_squared_error(targets[:, i], preds[:, i]) for i in range(len(HORIZONS))
        ]
        avg_mse = float(np.mean(mses))

        result = {
            "seed": seed,
            "dataset": dataset_key,
            "modality": modality,
            "fusion_type": fusion_type,
            "avg_mse": avg_mse,
        }
        for i, hrz in enumerate(HORIZONS):
            result[f"mse_{hrz}"] = mses[i]
        all_results.append(result)

        for i, hrz in enumerate(HORIZONS):
            all_item_results.append(pd.DataFrame({
                "seed": seed,
                "dataset": dataset_key,
                "modality": modality,
                "fusion_type": fusion_type,
                "ticker": item_ids["ticker"],
                "date": item_ids["date"],
                "horizon": hrz,
                "pred": preds[:, i],
                "target": targets[:, i],
                "sq_err": (preds[:, i] - targets[:, i]) ** 2,
            }))

    results_df = pd.DataFrame(all_results)


    if fusion_type == "residual_gated":
        csv_name = f"results_{dataset_key}_{modality}.csv"
    else:
        csv_name = f"results_{dataset_key}_{modality}_{fusion_type}.csv"

    results_df.to_csv(RESULTS_DIR / csv_name, index=False)

    item_results_df = pd.concat(all_item_results, ignore_index=True)

    return results_df, item_results_df

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
# The configurations reported in the paper: (dataset, QA representation, model)
PROPOSED = ("text_audio_prompt2", "residual_gated")
TABLE_III_MAEC16 = [
    ("text_only", "residual_gated"),            # Past Volatility + QA Text
    ("audio_only", "residual_gated"),           # Past Volatility + QA Audio
    ("text_audio_prompt1", "residual_gated"),   # Past Volatility + QA Text + Audio (General Prompt)
    PROPOSED,                                   # Past Volatility + QA Text + Audio (Volatility Prompt)
    ("chunk_text_audio", "residual_gated"),     # Past Volatility + Transcript Chunks + Audio
    ("text_audio_prompt2", "concat"),           # Past Volatility + QA Text + Audio (Concat)
    ("text_audio_prompt2", "qa_only"),          # QA Text + Audio (QA Only)
]
EXPERIMENTS = ([("maec15", *PROPOSED)] + [("maec16", m, f) for m, f in TABLE_III_MAEC16] + [("ec", *PROPOSED)])

experiment_config_df = pd.DataFrame(EXPERIMENTS, columns=["dataset", "modality", "fusion_type"])
experiment_config_df

In [ ]:
# [SETUP] Run mode "check": only the proposed model, on CHECK_DATASETS. In the other modes this cell changes nothing.
if RUN_MODE == "check":
    EXPERIMENTS = [e for e in EXPERIMENTS if e[0] in CHECK_DATASETS and tuple(e[1:]) == PROPOSED]
    assert EXPERIMENTS, "CHECK_DATASETS names no dataset of the paper"
    experiment_config_df = pd.DataFrame(EXPERIMENTS, columns=["dataset", "modality", "fusion_type"])

experiment_config_df

In [ ]:
# [EXPERIMENT] The paper's code, unchanged.
all_results = []
all_item_results = []
_qa_cache = {}

for _, cfg in experiment_config_df.iterrows():
    dataset_key = cfg["dataset"]
    modality = cfg["modality"]
    fusion_type = cfg["fusion_type"]
    name = f"{dataset_key}_{modality}_{fusion_type}"

    cache_key = (dataset_key, modality)
    if cache_key not in _qa_cache:
        _qa_cache[cache_key] = {
            "train": get_data_and_qa_for(dataset_key, "train", modality),
            "dev": get_data_and_qa_for(dataset_key, "dev", modality),
            "test": get_data_and_qa_for(dataset_key, "test", modality),
        }
    train_df, train_qas = _qa_cache[cache_key]["train"]
    dev_df, dev_qas = _qa_cache[cache_key]["dev"]
    test_df, test_qas = _qa_cache[cache_key]["test"]

    result_df, item_result_df = run_experiment(
        name=name,
        dataset_key=dataset_key,
        modality=modality,
        fusion_type=fusion_type,
        train_qas=train_qas,
        dev_qas=dev_qas,
        test_qas=test_qas,
        train_df=train_df,
        dev_df=dev_df,
        test_df=test_df,
        needs_vol_pretrain=(fusion_type == "residual_gated"),
    )
    all_results.append(result_df)
    all_item_results.append(item_result_df)

all_results_df = pd.concat(all_results, ignore_index=True)
all_results_df.to_csv(RESULTS_DIR / "all_results.csv", index=False)

all_item_losses_df = pd.concat(
    [*vol_only_item_results.values(), *all_item_results],
    ignore_index=True,
)
all_item_losses_df.to_csv(RESULTS_DIR / "item_losses.csv", index=False)

## 6. Our results in Tables I and III

Every test call is scored. A QA-based model has no prediction for a call without QA units; there it uses the same seed's Past Volatility prediction (MAEC-15: 4 calls, MAEC-16: 8, EC: 1). Values are means over the 55 seeds; marks are one-sided seed-paired t-tests (`scripts/make_tables.py`).

In `"smoke"` mode the tables are produced only to test the scripts. In `"check"` mode the tables are skipped and the run is compared with the released predictions instead.

In [ ]:
# [EXPERIMENT] Our results in Tables I and III from this run's predictions: scripts/make_tables.py, unchanged, with the same
# arguments as in the paper's runs (started from Python instead of a shell command). Not run in "check" mode.
_items = str(RESULTS_DIR / "item_losses.csv")
if RUN_MODE != "check":
    _run = subprocess.run(
        [sys.executable, os.path.join(REPO_DIR, "scripts", "make_tables.py"),
         "--maec15", _items, "--maec16", _items, "--ec", _items, "--out", str(RESULTS_DIR / "tables")],
        capture_output=True, text=True, encoding="utf-8", env={**os.environ, "PYTHONIOENCODING": "utf-8"})
    print(_run.stdout)
    if _run.returncode != 0:
        raise RuntimeError("make_tables.py failed:\n" + _run.stderr)
    if RUN_MODE == "smoke":
        print("*** SMOKE TEST: the numbers above are meaningless (2 seeds, 2 epochs). ***")

In [ ]:
# [CHECK] "check" mode: this run's predictions against the released ones (same seed, call and horizon).
# Writes only its report. A single seed is not expected to match exactly across hardware; the verdict says
# whether the new seeds are inside the spread of the 55 released seeds.
def compare_with_released():
    import io, platform, zipfile

    KEYS = ["seed", "dataset", "modality", "fusion_type", "ticker", "date", "horizon"]
    _rd = dict(dtype={"ticker": str, "date": str, "fusion_type": str, "modality": str}, keep_default_na=False)

    new = pd.read_csv(RESULTS_DIR / "item_losses.csv", **_rd)
    rel = pd.concat(
        [pd.read_csv(os.path.join(REPO_DIR, "results", "predictions", f"{d}_predictions.csv.gz"), **_rd)
         for d in sorted(new["dataset"].unique())],
        ignore_index=True,
    )
    rel = rel.merge(new[["dataset", "modality", "fusion_type"]].drop_duplicates())   # the configurations tested
    rel_all = rel.copy()                                                              # all 55 released seeds
    rel_all["model"] = np.where(rel_all["modality"] == "vol_only", "Past Volatility", "Proposed")
    rel = rel[rel["seed"].isin(SEEDS)]

    m = new.merge(rel, on=KEYS, how="outer", suffixes=("_new", "_rel"), indicator=True)
    both = m[m["_merge"] == "both"].copy()
    both["model"] = np.where(both["modality"] == "vol_only", "Past Volatility", "Proposed")
    both["abs_diff"] = (both["pred_new"] - both["pred_rel"]).abs()

    out = io.StringIO()
    def say(*a):
        print(*a); print(*a, file=out)

    say("CHECK AGAINST THE RELEASED PREDICTIONS")
    say("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)",
        "| torch", torch.__version__, "| python", platform.python_version())
    say("proposed model on:", CHECK_DATASETS, "| seeds:", SEEDS)
    say()
    say("1. Same calls and targets")
    say("   predictions: new", len(new), "| released", len(rel), "| matched", len(both),
        "| only new", int((m["_merge"] == "left_only").sum()), "| only released", int((m["_merge"] == "right_only").sum()))
    _tdiff = float((both["target_new"] - both["target_rel"]).abs().max())
    say(f"   largest target difference: {_tdiff:.2e}")
    say()
    say("2. Predictions (new minus released, absolute)")
    for (d, mod), g in both.groupby(["dataset", "model"]):
        say(f"   {d:7s} {mod:16s} max {g['abs_diff'].max():.6f} | median {g['abs_diff'].median():.6f} | "
            f"within 0.001: {100 * (g['abs_diff'] < 1e-3).mean():.1f}% | within 0.01: {100 * (g['abs_diff'] < 1e-2).mean():.1f}%")
    say()
    say("3. MSE, mean over the test seeds (calls each model predicts)")
    rows = []
    for (d, mod), g in both.groupby(["dataset", "model"]):
        for tag, col in (("new", "sq_err_new"), ("released", "sq_err_rel")):
            h = g.groupby("horizon")[col].mean()
            rows.append({"dataset": d, "model": mod, "run": tag, "avg": h.mean(), **{f"{int(k)}d": v for k, v in h.items()}})
    tab = pd.DataFrame(rows)
    say(tab.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
    _num = [c for c in tab.columns if c not in ("dataset", "model", "run")]
    _d = (tab[tab.run == "new"].set_index(["dataset", "model"])[_num]
          - tab[tab.run == "released"].set_index(["dataset", "model"])[_num])
    say()
    say("   difference (new minus released):")
    say(_d.to_string(float_format=lambda v: f"{v:+.4f}"))
    say()
    say("4. Average MSE per seed")
    ps = (both.groupby(["dataset", "model", "seed", "horizon"])[["sq_err_new", "sq_err_rel"]].mean()
          .groupby(["dataset", "model", "seed"]).mean())
    ps["diff"] = ps["sq_err_new"] - ps["sq_err_rel"]
    say(ps.to_string(float_format=lambda v: f"{v:.5f}"))
    say()
    say("5. Gain of the proposed model over Past Volatility, on the calls with QA units (average MSE)")
    for d, g in both.groupby("dataset"):
        p = g[g.model == "Proposed"]
        if len(p) == 0:
            continue
        v = g[g.model == "Past Volatility"].merge(p[["seed", "ticker", "date", "horizon"]])
        f = lambda x, col: x.groupby("horizon")[col].mean().mean()
        say(f"   {d:7s} new {f(v, 'sq_err_new') - f(p, 'sq_err_new'):+.4f} | released {f(v, 'sq_err_rel') - f(p, 'sq_err_rel'):+.4f}")
    say()
    say("6. New seeds against the spread of the 55 released seeds (average MSE per seed)")
    say("   A single seed is not expected to match across hardware. Band: the released range widened by one")
    say("   standard deviation on each side (with 55 seeds, a sound new seed can fall just outside the bare range).")
    def _per_seed(df, col):
        return (df.groupby(["dataset", "model", "seed", "horizon"])[col].mean()
                .groupby(["dataset", "model", "seed"]).mean())
    _r = _per_seed(rel_all, "sq_err")
    _n = _per_seed(both, "sq_err_new")
    _inside = True
    for (d, mod), r in _r.groupby(["dataset", "model"]):
        n = _n.loc[(d, mod)]
        lo, hi, mu, sd = r.min(), r.max(), r.mean(), r.std()
        ok = bool(((n >= lo - sd) & (n <= hi + sd)).all())
        _inside &= ok
        say(f"   {d:7s} {mod:16s} released: mean {mu:.4f}, sd {sd:.4f}, range {lo:.4f} to {hi:.4f} | "
            f"new: {', '.join(f'{v:.4f}' for v in n)} | {'inside' if ok else 'OUTSIDE'} the band")
    _gain_ok = True
    for d, g in rel_all.groupby("dataset"):
        p = g[g.model == "Proposed"]
        if len(p) == 0:
            continue
        v = g[g.model == "Past Volatility"].merge(p[["seed", "ticker", "date", "horizon"]])
        rg = (v.groupby(["seed", "horizon"])["sq_err"].mean().groupby("seed").mean()
              - p.groupby(["seed", "horizon"])["sq_err"].mean().groupby("seed").mean())
        b = both[both.dataset == d]
        pn = b[b.model == "Proposed"]
        vn = b[b.model == "Past Volatility"].merge(pn[["seed", "ticker", "date", "horizon"]])
        ng = (vn.groupby(["seed", "horizon"])["sq_err_new"].mean().groupby("seed").mean()
              - pn.groupby(["seed", "horizon"])["sq_err_new"].mean().groupby("seed").mean())
        ok = bool(((ng >= rg.min() - rg.std()) & (ng <= rg.max() + rg.std())).all())
        _gain_ok &= ok
        say(f"   {d:7s} gain of Proposed  released: mean {rg.mean():+.4f}, range {rg.min():+.4f} to {rg.max():+.4f} | "
            f"new: {', '.join(f'{x:+.4f}' for x in ng)} | {'inside' if ok else 'OUTSIDE'} the band")
    say()
    _max_pred = float(both["abs_diff"].max())
    _max_mse = float(_d.abs().max().max())
    _same = len(both) == len(new) == len(rel) and _tdiff < 1e-6
    if not _same:
        verdict = "FAIL: the calls or targets differ from the released predictions."
    elif _max_pred < 1e-5:
        verdict = "EXACT: the predictions are the released ones."
    elif _inside and _gain_ok:
        verdict = (f"CONSISTENT: not identical (largest MSE difference {_max_mse:.4f} over these seeds), "
                   "but every new seed is inside the band of the 55 released seeds.")
    else:
        verdict = f"DIFFERENT: a new seed is outside the band of the 55 released seeds (largest MSE difference {_max_mse:.4f})."
    say("VERDICT:", verdict)

    both.drop(columns=["_merge"]).to_csv(RESULTS_DIR / "check_comparison.csv", index=False)
    with open(RESULTS_DIR / "check_report.txt", "w", encoding="utf-8") as f:
        f.write(out.getvalue())
    print()
    print("report:", RESULTS_DIR / "check_report.txt", "| every prediction:", RESULTS_DIR / "check_comparison.csv")


if RUN_MODE == "check":
    compare_with_released()

In [ ]:
# [CHECK] "smoke" mode: confirms that the run produced its files. Changes nothing.
if RUN_MODE == "smoke":
    _made = pd.read_csv(RESULTS_DIR / "item_losses.csv")
    _configs = _made[["dataset", "modality", "fusion_type"]].drop_duplicates()
    _expected = len(VOL_ONLY_DATASET_KEYS) + len(EXPERIMENTS)   # Past Volatility per dataset + the QA configurations
    assert len(_configs) == _expected and _made["seed"].nunique() == 2, "the smoke run did not produce every configuration"
    assert (RESULTS_DIR / "tables" / "table1_ours.csv").exists() and (RESULTS_DIR / "tables" / "table3_ablation_maec16.csv").exists()
    print("SMOKE TEST PASSED: every cell ran on this setup.")
    print(f"  device: {device} | predictions: {len(_made):,} rows | models trained: {len(_configs)} x 2 seeds")
    print(f"  files written to {RESULTS_DIR}")
    print('  Next: set RUN_MODE = "check" or "full" in the settings cell and run all cells again.')